# 🍅 CronusFarm - 작물(아티초크 등) AI 자동 학습 노트북

이 노트북은 **사용자님이 직접 API Key와 프로젝트 이름만 넣으면** 자동으로 YOLOv8 모델을 학습시키고, 결과를 다운로드할 수 있게 만들어진 전용 파일입니다.

### ⚠️ 시작하기 전 필수 설정
상단 메뉴에서 **[런타임] -> [런타임 유형 변경]** 을 클릭하고 하드웨어 가속기를 **T4 GPU**로 설정해 주세요!

## 1단계: 필수 프로그램(YOLOv8, Roboflow) 설치

In [ ]:
!pip install ultralytics roboflow

## 2단계: 데이터셋(사진들) 다운로드
사용자님의 계정 정보(`cronus-emeyr`)와 아티초크 프로젝트(`artichoke-seedlings-aoi61`)가 이미 자동으로 입력되어 있습니다. 그대로 실행(▶)만 누르세요!

**🚨 주의:** 만약 `Version not found` 에러가 발생한다면, Roboflow 사이트의 왼쪽 메뉴에서 **[Generate]** -> **[Create New Version]** -> **[Generate]** 버튼을 한 번 눌러주신 뒤에 다시 실행해주세요.

In [ ]:
from roboflow import Roboflow
import sys

rf = Roboflow(api_key="QU8LCl2TkMr40pHhNnd1")
project = rf.workspace("cronus-emeyr").project("artichoke-seedlings-aoi61")

try:
    version = project.version(1)
    dataset = version.download("yolov8")
    print("✅ 다운로드 완료! 저장 위치:", dataset.location)
except Exception as e:
    print("\n❌ [오류 발생] 데이터셋 버전(Version)이 아직 생성되지 않았습니다!")
    print("Roboflow 사이트로 돌아가셔서 왼쪽 메뉴의 [Generate] 버튼을 누르고 버전을 1개 만들어주세요.")
    print("에러 상세내용:", e)
    sys.exit(1)

## 3단계: 인공지능 학습 시작! (가장 오래 걸리는 단계)
사진들을 50번(epochs=50) 반복해서 보면서 아티초크가 어떻게 생겼는지 학습합니다.

In [ ]:
from ultralytics import YOLO

# YOLOv8s 모델 불러오기 (가장 무난한 크기)
model = YOLO('yolov8s.pt')

# 학습 시작! (데이터셋 경로에 맞춰 자동으로 실행됩니다)
results = model.train(data=f"{dataset.location}/data.yaml", epochs=50, imgsz=640, batch=16)

## 4단계: 완성된 모델(.onnx) 변환 및 다운로드
학습이 끝난 최고 성능의 모델(`best.pt`)을 Hailo 칩 등에서 사용하기 좋게 ONNX 형식으로 변환 후, 내 컴퓨터로 다운로드합니다.

In [ ]:
import shutil
from google.colab import files
import os

best_model_path = 'runs/detect/train/weights/best.pt'

if os.path.exists(best_model_path):
    print("✅ 학습된 모델을 찾았습니다. ONNX로 변환합니다...")
    # 1. 모델을 ONNX 형식으로 내보내기
    trained_model = YOLO(best_model_path)
    trained_model.export(format='onnx')

    # 2. 내 컴퓨터로 완성된 모델 다운로드
    onnx_path = 'runs/detect/train/weights/best.onnx'
    target_path = '/content/cronusfarm_custom_model.onnx'
    shutil.copy(onnx_path, target_path)
    print("✅ 다운로드를 시작합니다!")
    files.download(target_path)
else:
    print("❌ 오류: 학습된 모델을 찾을 수 없습니다. 위 단계들이 정상적으로 끝났는지 확인해주세요.")